# Surrogate Models for Protein Sequence Activity Prediction
This notebook demonstrates how to train and evaluate surrogate models for protein fitness prediction.

The workflow is organized into four model families:
1. One-hot encoding based supervised models
2. PLM embedding based supervised models
3. Zero-shot PLM scoring methods
4. PLM fine-tuning methods (regression and contrastive)

We use ESMC as the primary protein language model, while compatible ESM2 variants can also be used. The example dataset is the cleaned GB1 fitness dataset in `../data/dataset_gb1.csv`.

In [ ]:
import sys
sys.path.append('..')

In [ ]:
import os
from tqdm import tqdm
import numpy as np
import pandas as pd

In [ ]:
from src.utils import helper
from src.models import top_model as topmodel
from src.models.embeddings import one_hot_encode

In [ ]:
from src.esm.esmc import ESMCLM
from src.models.finetuning import ESMCLoraRegression, ESMCConFit

## 1. Load and Inspect Data
We first load the GB1 dataset and inspect its columns. This dataset already contains predefined splits for train, validation, and test partitions.

In [ ]:
data_path = '../data'

In [ ]:
file = os.path.join(data_path, 'dataset_gb1.csv')
df = pd.read_csv(file)

In [ ]:
df.head()

The helper below creates boolean masks for train/validation/test from `split_id`.
- `split_id == 2`: train
- `split_id == 1`: validation
- `split_id == 0`: test

For GB1, this follows the common three-mutations-vs-rest split protocol. Set `omit_zero=True` to exclude zero-fitness samples from training when needed.

In [ ]:
def get_split_mask(df, omit_zero=False):
    # GB1 split convention: 2=train, 1=val, 0=test
    if omit_zero:
        train_mask = (df['split_id'] == 2) & (df['fitness_raw'] != 0)
    else:
        train_mask = (df['split_id'] == 2)

    val_mask = df['split_id'] == 1
    test_mask = df['split_id'] == 0

    return train_mask, val_mask, test_mask

## 2. One-Hot Encoding Based Surrogate Models
In this section, each sequence is represented with one-hot features and used with classical supervised regressors.

These models are lightweight baselines and provide a strong reference point before moving to PLM-based representations.

### 2.1 Build One-Hot Features and Data Splits
We one-hot encode sequences and then apply the split masks to produce `X_train`, `X_val`, `X_test` and matching labels.

In [ ]:
# One-hot encode each sequence into a fixed-size feature vector
embeddings = one_hot_encode(df['seq'])

In [ ]:
# Build split masks and select feature/label arrays for supervised training
train_mask, val_mask, test_mask = get_split_mask(df, omit_zero=False)

X_train = embeddings[train_mask]
X_val = embeddings[val_mask]
X_test = embeddings[test_mask]

y_train = df.loc[train_mask, 'fitness_log'].to_numpy().astype(np.float32)
y_val = df.loc[val_mask, 'fitness_log'].to_numpy().astype(np.float32)
y_test = df.loc[test_mask, 'fitness_log'].to_numpy().astype(np.float32)

In [ ]:
print(f'train {train_mask.sum()} val {val_mask.sum()} test {test_mask.sum()}')

### 2.2 Ridge Regression
Ridge regression is a linear baseline with L2 regularization. It is fast to train and often surprisingly competitive on one-hot features.

In [ ]:
surrogate = topmodel.RidgeSurrogate(alpha=1.0)
surrogate.trainmodel(X=X_train, y=y_train, val=(X_val, y_val))

In [ ]:
y_test_pred = surrogate.predict(X_test)

### 2.3 Random Forest
Random Forest captures nonlinear interactions between positions without requiring feature scaling, making it a useful nonparametric baseline.

In [ ]:
surrogate = topmodel.RFSurrogate()
surrogate.trainmodel(X=X_train, y=y_train, val=(X_test, y_test))

In [ ]:
y_test_pred = surrogate.predict(X_test)

### 2.4 Multi-Layer Perceptron (MLP)
The MLP baseline adds learnable nonlinear transformations over one-hot inputs. Hyperparameters below define layer sizes, optimization, and stopping behavior.

In [ ]:
print(f'input layer shape: {X_train.shape[1]}')

In [ ]:
# MLP hyperparameters for one-hot features
config={'layers': [5300, 512, 1], 
        'epoch': 100, 
        'batch_size': 16,
        'patience': 100,
        'early_stopping': False,
        'lr': 1e-3,
        'print_every_n_epoch': 10,
        'debug': True}
surrogate = topmodel.MLPSurrogate(config=config)
surrogate.trainmodel(X=X_train, y=y_train, val=(X_test, y_test))

In [ ]:
y_test_pred = surrogate.predict(X_test)

## 3. PLM Embedding Based Surrogate Models
Here, we replace one-hot features with embeddings from a pretrained protein language model (PLM).
Using PLM features often improves prediction quality by transferring evolutionary and structural signal from large-scale pretraining.

### 3.1 Extract PLM Embeddings and Build Splits
We initialize an ESMC backbone, compute sequence embeddings, and prepare train/validation/test arrays.
`embedding_choice='mean'` pools token embeddings, while `'concat'` flattens token representations into a larger feature vector.

In [ ]:
base_model = ESMCLM(name='esmc_600m', device='gpu')

In [ ]:
embedding_choice = 'mean' # options: 'mean' or 'concat'

# Mean pooling gives compact embeddings; concat keeps token-level detail at higher dimensionality
if embedding_choice == 'mean':
    embeddings = base_model.get_embeddings_mean(df['seq'])
elif embedding_choice == 'concat':
    embeddings = base_model.get_embeddings_flatten(df['seq'])
else:
    raise ValueError('model not found')

In [ ]:
# Reuse the same split strategy for PLM-derived embeddings
train_mask, val_mask, test_mask = get_split_mask(df, omit_zero=False)

X_train = embeddings[train_mask]
X_val = embeddings[val_mask]
X_test = embeddings[test_mask]

y_train = df.loc[train_mask, 'fitness_log'].to_numpy().astype(np.float32)
y_val = df.loc[val_mask, 'fitness_log'].to_numpy().astype(np.float32)
y_test = df.loc[test_mask, 'fitness_log'].to_numpy().astype(np.float32)

In [ ]:
print(f'train {train_mask.sum()} val {val_mask.sum()} test {test_mask.sum()}')

### 3.2 Ridge Regression on PLM Embeddings
A linear model on top of PLM embeddings is a strong and efficient baseline for transfer learning.

In [ ]:
surrogate = topmodel.RidgeSurrogate(alpha=1.0)
surrogate.trainmodel(X=X_train, y=y_train, val=(X_test, y_test))

In [ ]:
y_test_pred = surrogate.predict(X_test)

### 3.3 Random Forest on PLM Embeddings
This variant combines pretrained embeddings with a nonlinear tree ensemble to model complex sequence-function effects.

In [ ]:
surrogate = topmodel.RFSurrogate()
surrogate.trainmodel(X=X_train, y=y_train, val=(X_test, y_test))

In [ ]:
y_test_pred = surrogate.predict(X_test)

### 3.4 MLP on PLM Embeddings
An MLP head can further adapt PLM features to the target fitness landscape through supervised nonlinear mapping.

In [ ]:
print(f'input layer shape: {X_train.shape[1]}')

In [ ]:
config={'layers': [1152, 512, 1], 
        'epoch': 300, 
        'batch_size': 16,
        'patience': 200,
        'early_stopping': True,
        'lr': 1e-4,
        'print_every_n_epoch': 10,
        'debug': True}
surrogate = topmodel.MLPSurrogate(config=config)
surrogate.trainmodel(X=X_train, y=y_train, val=(X_test, y_test))

In [ ]:
y_test_pred = surrogate.predict(X_test)

## 4. Zero-Shot PLM Scoring
Zero-shot methods score variants directly from a pretrained PLM, without supervised training on GB1 labels.

Available scoring modes in this workflow:
- `wt_marginal`: compares mutant against wild-type marginals
- `masked_marginal`: masked-token based mutation scoring
- `pseudolikelihood`: autoregressive-style sequence score proxy

These scores can be evaluated directly against experimental fitness on train/validation/test partitions.

In [ ]:
base_model = ESMCLM(name='esmc_600m', device='gpu')

In [ ]:
wt_sequence = helper.read_fasta(os.path.join(data_path, 'GB1_WT.fasta'), mode='str')[0]

In [ ]:
zero_shot_method = 'masked_marginal'

In [ ]:
# Compute zero-shot scores for each mutant sequence
y_pred = []
for i, row in tqdm(df.iterrows()):
    mt_sequence = row['seq']

    if zero_shot_method == 'wt_marginal':
        score, n_muts = base_model.get_wildtype_marginal(mt_sequence, wt_sequence)
        assert n_muts == row['n_mut']
    elif zero_shot_method == 'masked_marginal':
        score, n_muts = base_model.get_masked_marginal(mt_sequence, wt_sequence)
        assert n_muts == row['n_mut']
    elif zero_shot_method == 'pseudolikelihood':
        score = base_model.pseudolikelihood(mt_sequence)
    else:
        raise ValueError('method not found')

    y_pred.append(score)

y_pred = np.array(y_pred)

In [ ]:
train_mask, val_mask, test_mask = get_split_mask(df, omit_zero=False)

y = df['fitness_log'].to_numpy().astype(np.float32)

y_train = y[train_mask]
y_val = y[val_mask]
y_test = y[test_mask]

y_train_pred = y_pred[train_mask]
y_val_pred = y_pred[val_mask]
y_test_pred = y_pred[test_mask]

## 5. Regression Fine-Tuning of PLMs
In this section, we fine-tune a PLM with a supervised regression objective to predict continuous fitness values.

The default model shown is ESMC (`esmc_600m`), but the same training pattern can be adapted to ESM2-based backbones where implemented in `src/models`.

In [ ]:
model_choices = 'esmc'  # alternatives can include ESM2 where supported

# Regression fine-tuning configuration
config={'epoch': 50, 
        'batch_size': 8,
        'lambda': 0.1,
        'accumulate_batch_size': 32,
        'patience': 20,
        'early_stopping': False,
        'lr': 1e-3,
        'print_every_n_epoch': 1,
        'device': 'gpu'}

surrogate = ESMCLoraRegression(name='esmc_600m', config=config)

surrogate.print_trainable_parameters(surrogate)

In [ ]:
train_mask, val_mask, test_mask = get_split_mask(df, omit_zero=False)

df_train = df[train_mask]
df_val = df[val_mask]
df_test = df[test_mask]

y_train = df_train['fitness_log'].to_numpy().astype(np.float32)
y_val = df_val['fitness_log'].to_numpy().astype(np.float32)
y_test = df_test['fitness_log'].to_numpy().astype(np.float32)

In [ ]:
surrogate.trainmodel(df_train, df_val)

In [ ]:
y_test_pred = surrogate.predict(df_test['seq'])

## 6. Contrastive Fine-Tuning of PLMs
Contrastive fine-tuning learns representations that separate more- and less-functional variants relative to wild type.

This objective can improve ranking quality and generalization when absolute fitness labels are noisy or limited.

In [ ]:
model_choices = 'esmc'  # alternatives can include ESM2 where supported

# Contrastive fine-tuning configuration
config={'epoch': 60, 
        'batch_size': 32,
        'lambda': 0.1,
        'accumulate_batch_size': 32,
        'patience': 20,
        'early_stopping': False,
        'model_checkpoint': True,
        'lr': 5e-4,
        'print_every_n_epoch': 1,
        'use_seq_head': True,
        'device': 'gpu'}

surrogate = ESMCConFit(name='esmc_600m', config=config)

surrogate.print_trainable_parameters(surrogate)

In [ ]:
surrogate

In [ ]:
wt_sequence = helper.read_fasta(os.path.join(data_path, 'GB1_WT.fasta'), mode='str')[0]

In [ ]:
train_mask, val_mask, test_mask = get_split_mask(df, omit_zero=False)

df_train = df[train_mask]
df_val = df[val_mask]
df_test = df[test_mask]

y_train = df_train['fitness_log'].to_numpy().astype(np.float32)
y_val = df_val['fitness_log'].to_numpy().astype(np.float32)
y_test = df_test['fitness_log'].to_numpy().astype(np.float32)

In [ ]:
surrogate.trainmodel(df_train, wt_sequence, val=df_val)

In [ ]:
y_test_pred = surrogate.predict(df_test['seq'], wt_sequence)